# ML-09 - Validation and Research Claim Audit

I audit Week 5's page review model using the same data and Random Forest settings.

## 1. Two paper findings + my methodology questions

I use [*The State of AI-Driven SEO*](../../docs/flyrank-seo-research-march-2026.pdf), March 2026, pages 5, 6, 9, and 36. These questions help me understand what its comparisons can support.

**Finding #1 - The Anatomy of Growing Content (page 6).** Growing pages average 3,180 words and 184 days of age; declining pages average 2,311 words and 230 days. Growth and decline are defined by impression changes between two 30-day windows.

**My question:** Would the word-count gap remain among similar clients, topics, and page ages? Would it also appear if growth were measured in a later month? I read this as an observed link between page length and growth. Testing content changes would help assess the effect of adding useful detail.

**Finding #4 - The Freshness Multiplier (page 9).** For pages at least 365 days old, recently refreshed pages have 3.2 times the health score and 57 times the impressions of stale pages. Refresh status comes from time since the last update.

**My question:** Was refresh status recorded before the performance window? Were the groups similar in earlier traffic and client mix? Comparing their changes over time would help account for which pages the team chose to refresh. I read the reported gap as an observed comparison between refreshed and stale pages.

## 2. My model under an honest split (before/after)

I keep Week 5's label: April 1-30 impressions below 80% of March 2-31 impressions count as a decline. All five inputs come from March. I keep the Random Forest settings chosen in Week 5.

Week 5 already kept clients separate. I added the random "before" split for this comparison. The grouped "after" split repeats Week 5's client holdout. Both tests contain 13,560 pages.

Precision@50 is the share of the top 50 pages with an observed decline. Average precision measures the full ranking. The test decline rate is the reference for random ranking.

Run `w04_signal_audit.ipynb` first to create the March and April summaries in `work/outputs/`. This audit loads those summaries and checks their page keys and data coverage.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'skills/README.md').exists())
OUTPUTS = ROOT / 'work/outputs'
KEYS = ['client_hash_id', 'content_hash_id']
LABEL = 'is_declining_label'
WINDOWS = [('2026-03-02', '2026-03-31'), ('2026-04-01', '2026-04-30')]

def monthly_pages(start, end):
    cache = OUTPUTS / f'w04_audit_{start[:7]}_v1.parquet'
    if not cache.exists():
        raise FileNotFoundError('Run work/notebooks/w04_signal_audit.ipynb to create the monthly summaries.')
    frame = pd.read_parquet(cache)
    assert frame['duplicate_rows'].sum() == 0
    assert not frame.duplicated(KEYS).any()
    days = (pd.Timestamp(end) - pd.Timestamp(start)).days + 1
    return frame.loc[frame['gsc_days'].eq(days) & frame['missing_gsc_rows'].eq(0)]

march, april = [monthly_pages(*window) for window in WINDOWS]
eligible = march['impressions'].ge(100) & march['early15'].gt(0)
features = march.loc[eligible, KEYS +
    ['impressions', 'clicks', 'position_sum', 'early15', 'late15']].rename(
    columns={'impressions': 'impressions_feature_30d', 'clicks': 'clicks_feature_30d'})
impressions = features['impressions_feature_30d']
features['momentum_pct'] = 100 * (features['late15'] - features['early15']) / features['early15']
features['ctr_feature_30d_pct'] = 100 * features['clicks_feature_30d'] / impressions
features['weighted_position_feature_30d'] = (
    features['position_sum'] / impressions).where(features['position_sum'].gt(0))

FEATURES = ['impressions_feature_30d', 'clicks_feature_30d',
            'ctr_feature_30d_pct', 'weighted_position_feature_30d', 'momentum_pct']
targets = april[KEYS + ['impressions']].rename(columns={'impressions': 'target_impressions'})
model_frame = features[KEYS + FEATURES].merge(targets, on=KEYS, validate='one_to_one')
model_frame[LABEL] = (model_frame['target_impressions']
                      < 0.80 * model_frame['impressions_feature_30d']).astype(int)
model_frame = model_frame.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES)
print(f'March eligible pages: {len(features):,}')
print(f'Pages with a complete April outcome and five usable inputs: {len(model_frame):,}')
print(f'Observed April decline rate: {model_frame[LABEL].mean():.1%}')

March eligible pages: 62,397
Pages with a complete April outcome and five usable inputs: 50,641
Observed April decline rate: 47.3%


In [2]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# Prepare the same Week 5 inputs once for all fits.
X = model_frame[FEATURES].copy()
for col in ('impressions_feature_30d', 'clicks_feature_30d'):
    X[col] = np.log1p(X[col])
X['momentum_pct'] = X['momentum_pct'].clip(upper=200)
y, groups = model_frame[LABEL], model_frame['client_hash_id']
forest = RandomForestClassifier(n_estimators=200, max_depth=8,
    min_samples_leaf=50, random_state=7, n_jobs=-1)

def ranked_pages(frame, scores):
    return frame.assign(score=scores).sort_values(
        ['score', 'impressions_feature_30d'], ascending=False, kind='stable')

def fit_and_rank(train_idx, test_idx, columns=FEATURES):
    model = clone(forest).fit(X.iloc[train_idx][columns], y.iloc[train_idx])
    scores = model.predict_proba(X.iloc[test_idx][columns])[:, 1]
    return model, ranked_pages(model_frame.iloc[test_idx], scores)

def ranking_metrics(ranked):
    return {'Test decline rate': ranked[LABEL].mean(),
            'Average precision': average_precision_score(ranked[LABEL], ranked['score']),
            'Precision@50': ranked[LABEL].head(50).mean()}

group_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7)
group_train_idx, group_test_idx = next(group_split.split(X, groups=groups))
random_split = train_test_split(np.arange(len(X)), test_size=len(group_test_idx), random_state=7)
splits = {'Page random (before)': random_split,
          'Client holdout (after)': (group_train_idx, group_test_idx)}
results, runs = [], {}
for name, (train_idx, test_idx) in splits.items():
    runs[name] = fit_and_rank(train_idx, test_idx)
    train_groups, test_groups = groups.iloc[train_idx], groups.iloc[test_idx]
    results.append({'Split': name, 'Test pages': len(test_idx),
        'Train clients': train_groups.nunique(), 'Test clients': test_groups.nunique(),
        'Clients in both': len(set(train_groups) & set(test_groups)),
        **ranking_metrics(runs[name][1])})

split_results = pd.DataFrame(results)
display(split_results.round(3))
assert split_results.loc[1, 'Clients in both'] == 0

group_model, group_ranked = runs['Client holdout (after)']
group_test = model_frame.iloc[group_test_idx]
rule_match = group_test['impressions_feature_30d'].ge(500) & group_test['momentum_pct'].le(-20)
rule_scores = (-group_test['momentum_pct']).where(rule_match, 0)
rule_metrics = ranking_metrics(ranked_pages(group_test, rule_scores))
print(f"Week 4 rule Precision@50: {rule_metrics['Precision@50']:.2f}; "
      f"test decline rate: {rule_metrics['Test decline rate']:.3f}.")

,Split,Test pages,Train clients,Test clients,Clients in both,Test decline rate,Average precision,Precision@50
0,Page random (before),13560,32,29,29,0.471,0.707,0.92
1,Client holdout (after),13560,24,8,0,0.413,0.698,0.90


Week 4 rule Precision@50: 0.86; test decline rate: 0.413.


The random split reaches 0.92 Precision@50; the client holdout reaches 0.90. That is one page out of 50. Client mix and the different decline rates may also affect this comparison.

On the same held-out clients, the forest finds 45 declines in its top 50, versus 43 for the Week 4 rule. This supports further testing of the review order. I already saw this holdout in Week 5, so it remains exploratory. A later month still needs testing.

## 3. Leakage audit

**Timing:** Features use March 2-31; the outcome uses April 1-30. Both March windows end before April begins. March impressions also set the label's comparison level. They are known at prediction time, and this shared value can affect the results.

**Inputs:** The model uses March impressions, clicks, CTR, weighted position, and momentum. The code checks this feature list against outcome columns, IDs, health scores, and product flags. IDs support joins and splits.

**Scope:** These results describe pages with complete data in both months and clients held out during one period. A later-month test would help assess performance over time.

In [3]:
# Check actual model inputs; the loader and merge enforce page uniqueness.
forbidden = set(KEYS + [LABEL, 'target_impressions', 'trend_pct',
    'trend_direction', 'health_score', 'optimization_flag', 'baseline_score'])
assert not set(X.columns) & forbidden
assert list(group_model.feature_names_in_) == FEATURES
assert np.isfinite(X.to_numpy()).all()
assert pd.Timestamp(WINDOWS[0][1]) < pd.Timestamp(WINDOWS[1][0])
print('Input, page uniqueness, and timing checks passed; client overlap is zero above.')

display(pd.Series(group_model.feature_importances_, index=FEATURES,
    name='Forest importance').sort_values(ascending=False).round(3).to_frame())

# Four-feature comparison: measure momentum's contribution.
other_features = [col for col in FEATURES if col != 'momentum_pct']
_, ablated_ranked = fit_and_rank(group_train_idx, group_test_idx, other_features)
display(pd.DataFrame(
    [ranking_metrics(ranked) for ranked in (group_ranked, ablated_ranked)],
    index=['All March features', 'Four March features']).round(3))

# Audit check: use April's outcome to reveal the label.
leaky_score = 0.8 * group_test['impressions_feature_30d'] - group_test['target_impressions']
assert (leaky_score.gt(0).astype(int) == group_test[LABEL]).all()
leaky_metrics = ranking_metrics(ranked_pages(group_test, leaky_score))
assert leaky_metrics['Average precision'] == leaky_metrics['Precision@50'] == 1.0
print('April-outcome check: average precision and Precision@50 both equal 1.0.')

Input, page uniqueness, and timing checks passed; client overlap is zero above.


,Forest importance
momentum_pct,0.448
impressions_feature_30d,0.194
ctr_feature_30d_pct,0.178
weighted_position_feature_30d,0.099
clicks_feature_30d,0.081


,Test decline rate,Average precision,Precision@50
All March features,0.413,0.698,0.90
Four March features,0.413,0.531,0.56


April-outcome check: average precision and Precision@50 both equal 1.0.


In [4]:
# Reuse the grouped ranking for real error examples.
top50, rest = group_ranked.head(50), group_ranked.iloc[50:]
false_alarms = top50.loc[top50[LABEL].eq(0)].head(3).assign(case='False alarm')
misses = rest.loc[rest[LABEL].eq(1)].head(3).assign(case='Queue miss')
assert len(false_alarms) == len(misses) == 3
cases = pd.concat([false_alarms, misses], ignore_index=True)
cases['april_change_pct'] = 100 * (cases['target_impressions'] / cases['impressions_feature_30d'] - 1)
bins = {
    'March volume': ('impressions_feature_30d', [99, 499, 2999, 29999, np.inf],
                     ['100-499', '500-2,999', '3,000-29,999', '30,000+']),
    'March momentum': ('momentum_pct', [-np.inf, -50, -20, 0, 20, np.inf],
                       ['loss >50%', 'loss 20-50%', 'loss 0-20%', 'gain 0-20%', 'gain >20%']),
    'Position tier': ('weighted_position_feature_30d', [0, 3, 10, 20, np.inf],
                      ['1-3', '4-10', '11-20', '20+']),
    'April change': ('april_change_pct', [-np.inf, -20, 0, 20, np.inf],
                     ['decline >20%', 'decline 0-20%', 'gain 0-20%', 'gain 20%+']),
}
for name, (col, edges, labels) in bins.items():
    cases[name] = pd.cut(cases[col], edges, labels=labels, right=name != 'April change')
display(cases[['case', *bins]])
print(f'False alarms in the 50-page queue: {top50[LABEL].eq(0).sum()} of 50. '
      f'Observed declines outside it: {rest[LABEL].eq(1).sum():,}.')

,case,March volume,March momentum,Position tier,April change
0,False alarm,"3,000-29,999",loss >50%,4-10,decline 0-20%
1,False alarm,"30,000+",loss >50%,4-10,gain 0-20%
2,False alarm,"3,000-29,999",loss >50%,20+,decline 0-20%
3,Queue miss,"3,000-29,999",loss >50%,20+,decline >20%
4,Queue miss,"3,000-29,999",loss >50%,20+,decline >20%
5,Queue miss,"3,000-29,999",loss >50%,20+,decline >20%


False alarms in the 50-page queue: 5 of 50. Observed declines outside it: 5,552.


These are real test pages, shown as ranges. All three false alarms had March losses above 50%. Two had smaller April declines; one gained impressions. The three missed declines also had large March losses and average positions beyond 20. They fell outside the 50-page queue. I would check demand, tracking, and page context before acting.

March momentum has the highest importance (0.448). Removing it lowers average precision from 0.698 to 0.531 and Precision@50 from 0.90 to 0.56. This measures how much the model relies on March momentum.

## 4. Claim rewrite

**Week 5 wording:** "Random Forest's small lead makes it worth testing on a later period before replacing the rule."

**Safer rewrite:** In this March-to-April client holdout, I measured Precision@50 of 0.90 for the forest and 0.86 for the rule, with a decline base rate of 0.413. That is two more observed declines in a 50-page queue. This is directional evidence for decision support. I would test a later month before changing the queue. A refresh experiment would help measure whether acting on this queue improves outcomes.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.